In [ ]:
import os
os.environ["PINECONE_API_KEY"] = "KEY"
os.environ["PINECONE_ENVIRONMENT"] = "ENV"


In [ ]:
# %%
# Cell 1 — Colab-ready installs + core imports
# Run once at the top of the Colab notebook. This cell installs packages and downloads spaCy model.

# Colab apt + pip installs (quiet) — may take several minutes
!apt-get update -qq && apt-get install -y -qq libmagic1 tesseract-ocr poppler-utils redis-server
!pip install -q pymupdf pytesseract sentence_transformers pinecone transformers rank_bm25 jiwer langdetect openai-whisper gTTS pdf2image deep-translator redis tqdm faiss-cpu spacy evaluate datasets networkx ray rouge_score bert_score
!pip install -U gradio>=5.1.0

# Download spaCy model
import spacy
try:
    spacy.cli.download("en_core_web_sm")
except Exception:
    pass

# %%
# Cell 2 — Core imports & lightweight config
import os
import io
import time
import tempfile
import json
import hashlib
import traceback
from collections import defaultdict, Counter

import fitz
import redis
import numpy as np
from PIL import Image
import pytesseract
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import whisper
from gtts import gTTS
from tqdm import tqdm
import torch
import networkx as nx
import faiss
import ray
import evaluate
from deep_translator import GoogleTranslator
from langdetect import detect

# Pinecone client (optional) — user can enable by setting PINECONE_API_KEY
try:
    import pinecone
    from pinecone import PineconeException
except Exception:
    pinecone = None

# Basic config
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
INDEX_NAME = "docqa-index-upgraded"
print("Device:", DEVICE)

# %%
# Cell 3 — Utilities: caching, logging, PDF helpers
# Redis (optional) — starts a local client; Colab may need background start for redis-server
redis_client = None
try:
    redis_client = redis.Redis(host='localhost', port=6379, db=0)
    redis_client.ping()
    print("✅ Redis available")
except Exception:
    redis_client = None
    print("⚠️ Redis not available — continuing without cache")


def cache_set(key, val, ttl=3600):
    if redis_client:
        try:
            redis_client.setex(key, ttl, json.dumps(val))
        except Exception as e:
            print("⚠️ Redis set failed:", e)


def cache_get(key):
    if not redis_client:
        return None
    try:
        val = redis_client.get(key)
        return json.loads(val) if val else None
    except Exception:
        return None

# PDF helpers

def extract_pdf_metadata(path):
    try:
        doc = fitz.open(path)
        meta = doc.metadata or {}
        doc.close()
        return {
            "title": (meta.get("title") or "").strip(),
            "author": (meta.get("author") or "").strip(),
            "subject": (meta.get("subject") or "").strip(),
            "keywords": (meta.get("keywords") or "").strip()
        }
    except Exception as e:
        print("⚠️ metadata error:", e)
        return {"title": "", "author": "", "subject": "", "keywords": ""}


def extract_text_from_pdf(path_or_file) -> str:
    if hasattr(path_or_file, "read"):
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix='.pdf')
        tmp.write(path_or_file.read()); tmp.flush()
        path = tmp.name
    else:
        path = path_or_file
    doc = fitz.open(path)
    pages_text = []
    for page in doc:
        t = page.get_text().strip()
        if not t:
            pix = page.get_pixmap(dpi=200)
            img = Image.open(io.BytesIO(pix.tobytes()))
            t = pytesseract.image_to_string(img)
        pages_text.append(t)
    doc.close()
    return "\n".join(pages_text)

# Simple latency logger
LATENCY_LOG = []

def log_latency(stage, t0):
    t = time.time() - t0
    LATENCY_LOG.append({"stage": stage, "latency": t, "ts": time.time()})
    return t

# %%
# Cell 4 — Chunking & tokenizer utilities (token-aware fallback)
from transformers import AutoTokenizer
TOK = AutoTokenizer.from_pretrained("google/flan-t5-base", use_fast=True)


def chunk_text(text, max_tokens=400, overlap=60):
    if not text:
        return []
    try:
        tokens = TOK.encode(text, truncation=False)
        chunks = []
        start = 0
        while start < len(tokens):
            end = min(len(tokens), start + max_tokens)
            chunk = TOK.decode(tokens[start:end], skip_special_tokens=True).strip()
            if chunk:
                chunks.append(chunk)
            if end == len(tokens):
                break
            start = end - overlap
        return chunks
    except Exception:
        words = text.split()
        chunks = []
        i = 0
        step = max(1, max_tokens - overlap)
        while i < len(words):
            chunk = " ".join(words[i:i+max_tokens])
            chunks.append(chunk)
            i += step
        return chunks

# %%
# Cell 5 — Embedding model + BM25 + optional Pinecone / FAISS setup
# ✅ Upgraded: multilingual embeddings + GPU FAISS + BM25 with auto-translation + Pinecone

from sentence_transformers import SentenceTransformer
import numpy as np
import faiss, os, torch

# Embedding model (multilingual, 100+ langs)
embed_model = SentenceTransformer("intfloat/multilingual-e5-large", device=DEVICE)
TARGET_DIM = embed_model.get_sentence_embedding_dimension()


def embed_texts_batched(texts, batch_size=64):
    if not texts:
        return np.zeros((0, TARGET_DIM), dtype=np.float32)
    embs = embed_model.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        show_progress_bar=False,
        normalize_embeddings=True
    )
    return embs

# BM25 placeholder (created during indexing)
BM25_INDEX = None
BM25_CORPUS = None  # tokenized lists
BM25_TEXTS = None   # original texts list (for provenance)
BM25_LANG = "en"  # default corpus language, updated during indexing

from deep_translator import GoogleTranslator


def translate_text(text, target_lang="en"):
    """Translate text safely with GoogleTranslator (fallback = original)."""
    try:
        return GoogleTranslator(source="auto", target=target_lang).translate(text)
    except Exception:
        return text


def bm25_query(query, top_k=5):
    """
    Run BM25 query with multilingual support:
    - Detect language of corpus and query
    - Auto-translate query if needed
    """
    global BM25_INDEX, BM25_CORPUS, BM25_LANG, BM25_TEXTS
    if BM25_INDEX is None or BM25_CORPUS is None:
        return []

    try:
        query_lang = detect(query)
    except Exception:
        query_lang = "en"

    # Translate query into corpus language if different
    q_proc = query
    if query_lang != BM25_LANG:
        q_proc = translate_text(query, target_lang=BM25_LANG)

    scores = BM25_INDEX.get_scores(q_proc.split())
    top_ids = np.argsort(scores)[::-1][:top_k]

    results = []
    for i in top_ids:
        results.append({"text": BM25_TEXTS[i], "tokens": BM25_CORPUS[i], "score": float(scores[i])})
    return results

# FAISS index (on-disk in Colab workspace)
FAISS_INDEX = None
FAISS_ID_TO_META = {}


def create_faiss_index(dim=TARGET_DIM, use_gpu=True):
    """
    Create FAISS index.
    If GPU available and use_gpu=True, moves FAISS index to GPU for faster retrieval.
    """
    global FAISS_INDEX
    if FAISS_INDEX is not None:
        return FAISS_INDEX

    # CPU index
    index = faiss.IndexFlatIP(dim)  # inner-product on normalized vectors ~ cosine

    if use_gpu and torch.cuda.is_available():
        try:
            res = faiss.StandardGpuResources()
            index = faiss.index_cpu_to_gpu(res, 0, index)
            print("✅ FAISS GPU index enabled")
        except Exception as e:
            print("⚠️ FAISS GPU init failed, falling back to CPU:", e)

    FAISS_INDEX = index
    return FAISS_INDEX

# Pinecone helper (if key present in env var)
PC = None
if os.environ.get('PINECONE_API_KEY') and pinecone is not None:
    try:
        pinecone.init(api_key=os.environ.get('PINECONE_API_KEY'))
        PC = pinecone
        print("✅ Pinecone initialized")
    except Exception as e:
        print("⚠️ Pinecone init failed:", e)

# %%
# Cell 6 — Index builder (multi-doc + graph extraction + FAISS upsert + BM25 + entity index)
nlp = spacy.load("en_core_web_sm")

# Entity index: mapping entity_text -> set of chunk ids (for quick lookup / UI)
ENTITY_TO_CHUNKS = defaultdict(set)
TOP_ENTITIES = []  # computed after indexing
INDEXED_FILES = []  # human-friendly list of uploaded files (keeps order)


def build_index_from_pdfs(file_paths, embed_batch_size=128, recreate_faiss=True, use_pinecone=False):
    """
    Builds FAISS + BM25 from multiple PDFs. Also extracts named entities and builds
    ENTITY_TO_CHUNKS for UI and graph expansion.
    """
    global BM25_INDEX, BM25_CORPUS, BM25_TEXTS, BM25_LANG, FAISS_INDEX, FAISS_ID_TO_META, ENTITY_TO_CHUNKS, TOP_ENTITIES, INDEXED_FILES

    # reset
    BM25_INDEX = None
    BM25_CORPUS = []
    BM25_TEXTS = []
    FAISS_ID_TO_META = {}
    ENTITY_TO_CHUNKS = defaultdict(set)
    TOP_ENTITIES = []
    INDEXED_FILES = []
    if recreate_faiss:
        FAISS_INDEX = None
    create_faiss_index()

    vectors = []
    all_chunks = []
    idx_counter = 0
    lang_votes = []

    for p in tqdm(file_paths, desc="Processing PDFs"):
        try:
            name = os.path.basename(p)
            INDEXED_FILES.append(name)
            meta_info = extract_pdf_metadata(p)
            if not any(meta_info.values()):
                meta_info['title'] = os.path.splitext(name)[0]

            txt = extract_text_from_pdf(p)
            chunks = chunk_text(txt, max_tokens=400, overlap=60)

            # detect language of this document (sample)
            try:
                doc_lang = detect(txt[:5000])
                lang_votes.append(doc_lang)
            except Exception:
                pass

            # build small KG for this doc and extract entities from a larger window
            doc_nlp = nlp(txt[:20000])  # process first N chars for speed
            G = nx.Graph()
            for ent in doc_nlp.ents:
                # normalize entity text
                ent_text = ent.text.strip()
                if ent_text:
                    G.add_node(ent_text, label=ent.label_)
            # naive edges by co-occurrence within sentence window
            for sent in doc_nlp.sents:
                ents = [e.text.strip() for e in sent.ents if e.text.strip()]
                for i in range(len(ents)):
                    for j in range(i+1, len(ents)):
                        G.add_edge(ents[i], ents[j])

            # index meta info as a chunk and tag source
            for k, v in meta_info.items():
                if v:
                    text = f"{k.capitalize()}: {v}"
                    chunks.insert(0, text)

            # embeddings in batches
            for i in range(0, len(chunks), embed_batch_size):
                batch_chunks = chunks[i:i+embed_batch_size]
                embs = embed_texts_batched(batch_chunks, batch_size=embed_batch_size)
                for j, (c, e) in enumerate(zip(batch_chunks, embs)):
                    cid = f"{name}__{i+j}"
                    meta = {"chunk_id": cid, "source": name, "text": c, "graph": nx.to_dict_of_lists(G), "source_file": name}
                    # add to FAISS
                    FAISS_INDEX.add(np.expand_dims(e.astype(np.float32), axis=0))
                    FAISS_ID_TO_META[idx_counter] = meta
                    idx_counter += 1
                    all_chunks.append(meta)
                    BM25_CORPUS.append(c.split())
                    BM25_TEXTS.append(c)

                    # associate entities in this chunk (cheap heuristic: look for node mentions)
                    for node in list(G.nodes())[:20]:
                        # if node appears in chunk text (case-insensitive), map it
                        if node.lower() in c.lower():
                            ENTITY_TO_CHUNKS[node].add(cid)

        except Exception as exc:
            print(f"⚠️ Failed processing {p}: {exc}")
            traceback.print_exc()
            raise

    # build BM25 index
    if BM25_CORPUS:
        BM25_INDEX = BM25Okapi(BM25_CORPUS)

    # set BM25_LANG to most common detected language across docs
    if lang_votes:
        BM25_LANG = Counter(lang_votes).most_common(1)[0][0]
    else:
        BM25_LANG = "en"

    # compute TOP_ENTITIES for UI (most frequent by number of chunks)
    ent_counts = [(ent, len(chunks)) for ent, chunks in ENTITY_TO_CHUNKS.items()]
    ent_counts.sort(key=lambda x: x[1], reverse=True)
    TOP_ENTITIES = ent_counts[:50]  # top 50 entities (text, count)

    print(f"✅ Indexed {len(all_chunks)} chunks (FAISS size: {FAISS_INDEX.ntotal}) — BM25_LANG={BM25_LANG}")
    return all_chunks, BM25_INDEX, BM25_TEXTS

# %%
# Cell 7 — Retriever: Dense + Sparse + Graph neighbor expansion
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=DEVICE)
print("✅ Reranker loaded")


def retrieve_candidates(query, top_k_sem=10, top_k_bm=10, combined_k=50, sem_weight=0.6, bm_weight=0.4):
    # embed query
    q_emb = embed_texts_batched([query], batch_size=1)[0].astype(np.float32)

    # FAISS search (dense)
    D, I = FAISS_INDEX.search(np.expand_dims(q_emb, axis=0), top_k_sem)
    pine_matches = []
    for score, idx in zip(D[0], I[0]):
        if idx < 0:
            continue
        meta = FAISS_ID_TO_META.get(int(idx), {})
        pine_matches.append({'meta': meta, 'pine_score': float(score)})

    # BM25
    bm_candidates = []
    if BM25_INDEX and BM25_TEXTS:
        # translate query into corpus language if needed
        try:
            q_lang = detect(query)
        except Exception:
            q_lang = "en"
        q_proc = query
        if q_lang != BM25_LANG:
            q_proc = translate_text(query, target_lang=BM25_LANG)
        bm_scores = BM25_INDEX.get_scores(q_proc.split())
        bm_scores = np.array(bm_scores, dtype=float)
        if bm_scores.size:
            top_bm_idx = list(np.argsort(-bm_scores)[:top_k_bm])
            for idx in top_bm_idx:
                text = BM25_TEXTS[idx]
                bm_candidates.append({
                    'meta': {'chunk_id': f'bm25__{idx}', 'source': 'bm25', 'text': text, 'source_file': None},
                    'bm25_score': float(bm_scores[idx])
                })

    # merge
    candidates = {}
    for p in pine_matches:
        m = p['meta']
        key = m.get('chunk_id') or (m.get('text','')[:120])
        candidates[key] = {
            'text': m.get('text',''),
            'source': m.get('source','faiss'),
            'chunk_id': m.get('chunk_id', key),
            'pine_score': p.get('pine_score',0.0),
            'bm25_score': 0.0,
            'meta_graph': m.get('graph'),
            'source_file': m.get('source_file')
        }
    for b in bm_candidates:
        m = b['meta']
        key = m.get('chunk_id')
        if key in candidates:
            candidates[key]['bm25_score'] = b.get('bm25_score', 0.0)
        else:
            candidates[key] = {
                'text': m.get('text',''),
                'source': m.get('source','bm25'),
                'chunk_id': key,
                'pine_score': 0.0,
                'bm25_score': b.get('bm25_score', 0.0),
                'meta_graph': None,
                'source_file': m.get('source_file')
            }

    # combine scores
    for c in candidates.values():
        c['combined_score'] = sem_weight * c.get('pine_score',0.0) + bm_weight * c.get('bm25_score',0.0)

    # limit
    sorted_by_combined = sorted(candidates.items(), key=lambda kv: kv[1]['combined_score'], reverse=True)
    limited = dict(sorted_by_combined[:combined_k])

    # graph expansion: add neighbors from meta_graph if present
    expanded = dict(limited)
    for k, v in limited.items():
        gdict = v.get('meta_graph')
        if gdict:
            # naive expansion: pick top nodes and find other chunks that mention them
            for node in list(gdict.keys())[:5]:
                # search BM25 corpus for node mention
                for i, tokens in enumerate(BM25_CORPUS or []):
                    if node.lower() in " ".join(tokens).lower():
                        key2 = f'bm25__{i}'
                        if key2 not in expanded:
                            text = " ".join(tokens)
                            expanded[key2] = {
                                'text': text, 'source': 'graph_expand', 'chunk_id': key2,
                                'pine_score': 0.0, 'bm25_score': 0.0, 'combined_score': 0.0,
                                'source_file': None
                            }
    # rerank
    rerank_pairs = [[query, c['text']] for c in expanded.values() if c['text']]
    rerank_scores = reranker.predict(rerank_pairs) if rerank_pairs else []
    if rerank_scores is not None and len(rerank_scores) > 0:
        if not isinstance(rerank_scores, list):
            rerank_scores = rerank_scores.tolist()
    for (key, cand), score in zip(expanded.items(), rerank_scores):
        cand['rerank_score'] = float(score)

    sorted_cands = sorted(expanded.values(), key=lambda x: x.get('rerank_score', x.get('combined_score',0.0)), reverse=True)
    return sorted_cands

# %%
# Cell 8 — Multi-document reasoning & summarization
# Lightweight summarizer (use Flan-T5 summarization prompt)
qa_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
qa_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")
qa_pipe = pipeline("text2text-generation", model=qa_model, tokenizer=qa_tokenizer, device=0 if torch.cuda.is_available() else -1)


def summarize_text(text, max_length=128):
    prompt = f"Summarize the following text in a concise paragraph:\n\n{text}"
    out = qa_pipe(prompt, max_length=max_length, truncation=True)[0]["generated_text"].strip()
    return out


def multi_doc_answer(question, candidates, max_docs=5):
    # take top unique sources
    top = []
    seen_src = set()
    for c in candidates:
        src = c.get('source')
        if src not in seen_src:
            top.append(c)
            seen_src.add(src)
        if len(top) >= max_docs:
            break

    summaries = [summarize_text(c.get('text',''), max_length=80) for c in top]
    combined = "\n\n".join([f"[{c['chunk_id']}] {s}" for c, s in zip(top, summaries)])

    prompt = (
        "You are a helpful assistant. Use ONLY the CONTEXTS below (summaries of documents) to answer the question. "
        "If the exact answer is not present, say 'I don't know'.\n\n"
        f"CONTEXTS:\n{combined}\n\nQUESTION: {question}\nANSWER:"
    )
    out = qa_pipe(prompt, max_length=256)[0]["generated_text"].strip()
    return out, top

# %%
# Cell 9 — Question classification (kept & slightly improved)
CLASS_PROTOTYPES = {
    "metadata": [
        "A question asking for document metadata like title, author, subject, or keywords. Example: 'What is the title of the document?'",
        "Find the author's name or the document title. Example: 'Who is the author?'"
    ],
    "factual": [
        "A factual lookup question that expects an exact span from the document. Example: 'When was the report published?'",
        "Retrieve concrete facts from the text. Example: 'List the projects mentioned in the resume.'"
    ],
    "interpretive": [
        "An interpretive or opinion question that requires inference from the context. Example: 'Is she a good professional?'",
        "A subjective assessment based on the text. Example: 'Is this person experienced?'"
    ],
    "boolean": [
        "A yes/no question. Example: 'Is she a student?'",
        "A binary question expecting yes/no or short answer. Example: 'Does the resume show employment?'"
    ]
}


def classify_question_type(question):
    q = question.strip()
    q_lower = q.lower()

    metadata_words = {"title", "author", "subject", "keywords", "keyword", "published", "publisher"}
    yesno_start = ("is ", "are ", "was ", "were ", "do ", "does ", "did ", "has ", "have ", "can ", "could ", "will ", "would ")

    heuristic = None
    if any(w in q_lower for w in metadata_words):
        heuristic = "metadata"
    elif any(q_lower.startswith(w) for w in ["what", "when", "where", "who", "which"]):
        heuristic = "factual"
    elif q_lower.startswith(yesno_start):
        heuristic = "boolean"

    # fallback to reranker-based classification
    pairs = []
    mapping = []
    for label, protos in CLASS_PROTOTYPES.items():
        for p in protos:
            pairs.append([q, p])
            mapping.append(label)
    try:
        vals = reranker.predict(pairs)
        acc = defaultdict(list)
        for lab, v in zip(mapping, vals):
            acc[lab].append(float(v))
        scores = {lab: float(np.mean(lst)) if lst else -1e6 for lab, lst in acc.items()}
        chosen = max(scores.items(), key=lambda x: x[1])[0]
    except Exception:
        chosen = heuristic or "interpretive"

    return {"label": chosen, "heuristic": heuristic}

# %%
# Cell 10 — Unified Answer function (English-only outputs + reasoning fallback + evaluation)

# Load evaluation metrics (keep them for testing/debug)
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

def reasoning_answer(question_en, candidates, qtype, top_k=5):
    """
    question_en: QUESTION ALREADY TRANSLATED TO ENGLISH (we process everything in English).
    Returns English string (always).
    """
    contexts = [c.get('text','') for c in candidates[:top_k] if c.get('text')]
    combined = "\n".join([f"- {c}" for c in contexts])

    if qtype == "boolean":
        prompt = (
            "You are a careful assistant. Use ONLY the CONTEXTS to answer the question. "
            "If contexts contain evidence, answer YES or NO followed by a one-line justification referencing the most relevant context. "
            "If not enough evidence, say 'I don't know'.\n\n"
            f"CONTEXTS:\n{combined}\n\nQUESTION: {question_en}\nANSWER:"
        )
    elif qtype == "interpretive":
        prompt = (
            "You are a thoughtful assistant. Use the CONTEXTS to provide an interpretive answer. "
            "Base your judgment on the contexts; if you must infer, clearly label it as 'inference'. "
            "If no relevant context, say 'Not enough information'.\n\n"
            f"CONTEXTS:\n{combined}\n\nQUESTION: {question_en}\nANSWER:"
        )
    else:
        prompt = (
            "You are a precise assistant. Use ONLY the CONTEXTS to answer the question. "
            "If the answer cannot be found in the contexts, say 'I don't know'.\n\n"
            f"CONTEXTS:\n{combined}\n\nQUESTION: {question_en}\nANSWER:"
        )

    out = qa_pipe(prompt, max_length=256)[0]["generated_text"].strip()
    return out, candidates[:top_k]


def answer_question(question, top_k=3, do_multi_hop=True):
    """
    - Translate incoming question to English (if needed) for processing.
    - Always return the answer in English.
    """
    t0 = time.time()

    # Detect language of question
    try:
        lang = detect(question)
    except Exception:
        lang = "en"

    # Translate to English for processing if needed (but do NOT translate output back)
    proc_q = question
    if lang != "en":
        try:
            proc_q = GoogleTranslator(source=lang, target="en").translate(question)
        except Exception:
            proc_q = question

    cls = classify_question_type(proc_q)
    candidates = retrieve_candidates(proc_q, top_k_sem=12, top_k_bm=12, combined_k=40)
    log_latency('retrieval', t0)

    if not candidates:
        return {
            "answer": "I don't know",
            "provenance": [],
            "classification": cls,
            "latency": time.time()-t0
        }

    # --- Metadata mode ---
    if cls['label'] == 'metadata':
        meta_fields = ["Title:", "Author:", "Subject:", "Keywords:"]
        meta_results = defaultdict(list)
        for c in candidates:
            text = c.get('text','')
            fname = c.get('source_file') or c.get('source') or 'unknown'
            for f in meta_fields:
                if text.startswith(f):
                    key = f[:-1]
                    val = text[len(f):].strip()
                    meta_results[key].append(f"{val} (from {fname})")

        ans_lines = []
        for f in meta_fields:
            key = f[:-1]
            vals = meta_results.get(key, [])
            if vals:
                ans_lines.append(f"{key}: " + "; ".join(vals))

        ans_text = "\n".join(ans_lines) if ans_lines else "I don't know"

        return {
            "answer": ans_text,
            "provenance": candidates[:top_k],
            "classification": cls,
            "latency": time.time()-t0
        }

    # --- Boolean or Interpretive ---
    if cls['label'] in ["boolean", "interpretive"]:
        out, top_sources = reasoning_answer(proc_q, candidates, cls['label'], top_k=5)
        return {
            "answer": out,
            "provenance": top_sources,
            "classification": cls,
            "latency": time.time()-t0
        }

    # --- Factual / Multi-hop ---
    if do_multi_hop:
        out, top_sources = multi_doc_answer(proc_q, candidates, max_docs=5)
    else:
        combined = "\n\n".join([f"[{c['chunk_id']}] {c.get('text','')}" for c in candidates[:top_k]])
        prompt = (
            "You are a helpful assistant. Use only the contexts to answer. "
            "If not present, say 'I don't know'.\n\n"
            f"CONTEXTS:\n{combined}\n\nQUESTION: {proc_q}\nANSWER:"
        )
        out = qa_pipe(prompt, max_length=256)[0]["generated_text"].strip()
        top_sources = candidates[:top_k]

    return {
        "answer": out,
        "provenance": top_sources,
        "classification": cls,
        "latency": time.time()-t0
    }


# %%
# Evaluation helper (kept)
def evaluate_answer(pred, gold):
    r = rouge.compute(predictions=[pred], references=[gold])
    b = bertscore.compute(predictions=[pred], references=[gold], lang="en")
    return {"rouge": r, "bertscore": b}

# %%
# Cell 11 — ASR + TTS (kept, Colab-friendly) — TTS forced to English outputs

asr_model = whisper.load_model("small")

def transcribe_audio(path):
    """Return transcription (language-agnostic) from audio using Whisper."""
    return asr_model.transcribe(path)["text"]

def synthesize_tts(text, lang="en"):
    """
    Synthesize TTS. We default to English output to match 'English-only' requirement.
    If you want other TTS languages, override lang when calling.
    """
    out_path = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3").name
    try:
        gTTS(text=text, lang=lang).save(out_path)
    except Exception:
        # fallback: save minimal silent file or raise a clear error
        gTTS(text=text, lang="en").save(out_path)
    return out_path

# %%
# Cell 12 — Modern Gradio DocQA UI (English-only answers, history, TTS, voice, analytics)
import gradio as gr
import traceback
import pandas as pd
import json
import time

HISTORY = []

# CSS for nicer UI
CUSTOM_CSS = """
#app-title {font-size:1.6rem; font-weight:700; margin-bottom:6px;}
.small-muted {color:#6b7280; font-size:0.9rem;}
.prov-box {font-family:monospace; font-size:0.9rem; white-space:pre-wrap;}
.gr-row {gap: 12px;}
"""

with gr.Blocks(css=CUSTOM_CSS) as demo:
    gr.HTML('<div id="app-title">🌍 Intelligent DocQA — English-only answers, multi-doc reasoning</div>')

    with gr.Row():
        # --- Left Sidebar: Upload + Index + Entities ---
        with gr.Column(scale=2):
            gr.Markdown("### 📂 Upload & Build Index")
            pdf_in = gr.File(file_count="multiple", file_types=[".pdf"], label="Upload PDFs")
            build_btn = gr.Button("Build Index", variant="primary")
            info = gr.Textbox(label="Index Status", interactive=False)
            file_list = gr.Textbox(label="Indexed files", interactive=False, lines=4)
            entities_table = gr.Dataframe(headers=["Entity", "Chunk count"], label="Top entities", interactive=False)

        # --- Main Column: Question + Answer + Provenance + History ---
        with gr.Column(scale=3):
            gr.Markdown("### ❓ Ask Questions")
            q_in = gr.Textbox(label="Ask a question (any language) — ENGLISH answer only")
            q_btn = gr.Button("Ask", variant="primary")

            with gr.Row():
                q_out = gr.Textbox(label="Answer (English)", lines=6)
                q_audio = gr.Audio(label="Answer (TTS)", interactive=False)

            with gr.Accordion("📖 Provenance & Evidence", open=False):
                q_prov = gr.Textbox(label="Provenance (JSON)", lines=12, interactive=False)

            with gr.Accordion("🕒 History (Latest Queries)", open=False):
                hist_box = gr.Textbox(label="History", lines=8, interactive=False)

    # --- Bottom Row: Voice + Analytics ---
    with gr.Row():
        with gr.Column(scale=2):
            v_in = gr.Audio(label="Ask by Voice", type="filepath")
            v_btn = gr.Button("Ask by Voice")
            v_text = gr.Textbox(label="Transcription", interactive=False)
        with gr.Column(scale=3):
            analytics = gr.Textbox(label="Latency / Debug", lines=6, interactive=False)

    # --- Helpers ---
    def _format_top_entities():
        if not TOP_ENTITIES:
            return pd.DataFrame([], columns=["Entity", "Chunk count"])
        rows = [(e, c) for e, c in TOP_ENTITIES]
        return pd.DataFrame(rows, columns=["Entity", "Chunk count"])

    # --- Callbacks ---
    def build_index_ui(files):
        if not files:
            return "No files provided", "", _format_top_entities().to_dict(), ""
        paths = [str(f) for f in files]
        metas, bm25_idx, bm25_corpus = build_index_from_pdfs(paths)
        info_msg = f"Indexed {len(metas)} chunks"
        file_list_str = "\n".join(INDEXED_FILES)
        df = _format_top_entities()
        return info_msg, file_list_str, df, info_msg

    def ask_text_ui(q):
        if not q:
            return "", None, "", "", ""
        t0 = time.time()
        try:
            res = answer_question(q)
            audio = synthesize_tts(res['answer'], lang="en") if res.get('answer') else None
            prov_json = json.dumps({
                "classification": res['classification'],
                "provenance": res['provenance']
            }, indent=2, ensure_ascii=False)
            HISTORY.append({"q": q, "res": res})
            hist_preview = "\n".join([f"{h['q']} -> {h['res']['answer'][:150]}" for h in HISTORY[-10:]][::-1])
            latency_log = json.dumps(LATENCY_LOG[-10:], indent=2)
            return res['answer'], audio, prov_json, latency_log, hist_preview
        except Exception as e:
            tb = traceback.format_exc()
            return "⚠️ Error", None, f"Error: {str(e)}", f"Traceback:\\n{tb}", ""

    def ask_voice_ui(audio_file):
        if not audio_file:
            return "", "", None, "", ""
        try:
            text = transcribe_audio(audio_file)
            res = answer_question(text)
            audio = synthesize_tts(res['answer'], lang="en") if res.get('answer') else None
            prov_json = json.dumps({
                "classification": res['classification'],
                "provenance": res['provenance']
            }, indent=2, ensure_ascii=False)
            HISTORY.append({"q": text, "res": res})
            hist_preview = "\n".join([f"{h['q']} -> {h['res']['answer'][:150]}" for h in HISTORY[-10:]][::-1])
            latency_log = json.dumps(LATENCY_LOG[-10:], indent=2)
            return text, res['answer'], audio, prov_json, latency_log, hist_preview
        except Exception as e:
            tb = traceback.format_exc()
            return "⚠️ Error in voice", "⚠️", None, f"Traceback:\\n{tb}", "", ""

    # --- Bind events ---
    build_btn.click(build_index_ui, inputs=[pdf_in], outputs=[info, file_list, entities_table, analytics])
    q_btn.click(ask_text_ui, inputs=[q_in], outputs=[q_out, q_audio, q_prov, analytics, hist_box])
    v_btn.click(ask_voice_ui, inputs=[v_in], outputs=[v_text, q_out, q_audio, q_prov, analytics, hist_box])

demo.launch(share=True)


# %%
# End of notebook
# Notes:
# - This upgraded notebook keeps your original pipeline design but adds: multi-doc reasoning, a small KG extraction
#   used for candidate expansion, FAISS-based dense retrieval (Colab-friendly), evaluation hooks, and a cleaner UI.
# - In Colab set PINECONE_API_KEY env var if you prefer Pinecone instead of FAISS.
# - Tweak hyperparameters (chunk sizes, top_k) for your dataset.


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libjemalloc2:amd64.
(Reading database ... 126441 files and directories currently installed.)
Preparing to unpack .../0-libjemalloc2_5.2.1-4ubuntu1_amd64.deb ...
Unpacking libjemalloc2:amd64 (5.2.1-4ubuntu1) ...
Selecting previously unselected package liblua5.1-0:amd64.
Preparing to unpack .../1-liblua5.1-0_5.1.5-8.1build4_amd64.deb ...
Unpacking liblua5.1-0:amd64 (5.1.5-8.1build4) ...
Selecting previously unselected package liblzf1:amd64.
Preparing to unpack .../2-liblzf1_3.6-3_amd64.deb ...
Unpacking liblzf1:amd64 (3.6-3) ...
Selecting previously unselected package lua-bitop:amd64.
Preparing to unpack .../3-lua-bitop_1.0.2-5_amd64.deb ...
Unpacking lua-bitop:amd64 (1.0.2-5) ...
Selecting previously unselected package lua-cjson:amd64.
Preparing to unpack .../4-

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device set to use cuda:0


100%|███████████████████████████████████████| 461M/461M [00:09<00:00, 51.2MiB/s]


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b3460d03d7e1d32712.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
